E5 — Improved document-wide chunking.

It uses the same training and validation datasets as E1, initializes XLM-R from the existing E1 checkpoint, and trains a new model with chunks selected across the entire document rather than taking only the first eight chunks.

It will:

Train E5 with document-wide chunk selection.
Use attention pooling over chunk embeddings.
Validate after every epoch and save the best checkpoint by macro-F1.
Generate a per-class precision/recall/F1 table and confusion matrix.
Save the tokenizer, model configuration, validation predictions and all artifacts.
Not run test inference or create a submission.

In [1]:
import os
import json
import random
import gc
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Dataset paths
BASE_PATH = "/kaggle/input/datasets/suchokkateryna/base-train-data"

TRAIN_PATH = f"{BASE_PATH}/train_transformer_40k.csv"
VAL_PATH = f"{BASE_PATH}/val_fixed.csv"

# E1 artifacts
E1_DIR = (
    "/kaggle/input/datasets/suchokkateryna/"
    "experiment1-artifacts/experiments/e1_chunked"
)

E1_MODEL_PATH = f"{E1_DIR}/model_state.pt"
E1_CONFIG_PATH = f"{E1_DIR}/arch_config.json"

# Output
OUT_DIR = "/kaggle/working/e5_chunking_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

# Labels
LABELS = ["human", "ai", "fusion"]

LABEL2ID = {
    "human": 0,
    "ai": 1,
    "fusion": 2
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Model
MODEL_NAME = "xlm-roberta-base"

MAX_LEN = 512
CONTENT_LEN = MAX_LEN - 2
STRIDE = 256
MAX_CHUNKS = 8

# Training
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4

ENCODER_LR = 1e-5
HEAD_LR = 5e-5
WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0

PATIENCE = 1

NUM_WORKERS = 2

USE_AMP = DEVICE.type == "cuda"

print("Configuration initialized.")

Device: cuda
Configuration initialized.


Cell 2 — Load datasets

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

TEXT_COL = "text"
LABEL_COL = "label"

assert TEXT_COL in train_df.columns
assert LABEL_COL in train_df.columns
assert TEXT_COL in val_df.columns
assert LABEL_COL in val_df.columns

X_train = train_df[TEXT_COL].fillna("").astype(str).tolist()
X_val = val_df[TEXT_COL].fillna("").astype(str).tolist()

y_train_raw = train_df[LABEL_COL].astype(str).str.lower().str.strip()
y_val_raw = val_df[LABEL_COL].astype(str).str.lower().str.strip()

assert set(y_train_raw.unique()).issubset(LABEL2ID)
assert set(y_val_raw.unique()).issubset(LABEL2ID)

y_train = y_train_raw.map(LABEL2ID).to_numpy(dtype=np.int64)
y_val = y_val_raw.map(LABEL2ID).to_numpy(dtype=np.int64)

print("Train:", len(X_train))
print("Validation:", len(X_val))

print("\nTrain distribution:")
display(y_train_raw.value_counts())

print("\nValidation distribution:")
display(y_val_raw.value_counts())

if "id" in train_df.columns and "id" in val_df.columns:
    overlap = (
        set(train_df["id"].astype(str))
        & set(val_df["id"].astype(str))
    )
    assert len(overlap) == 0, "Train/validation IDs overlap!"

print("Dataset loaded.")

Train: 40000
Validation: 40853

Train distribution:


label
ai        17909
fusion    12290
human      9801
Name: count, dtype: int64


Validation distribution:


label
ai        18291
fusion    12552
human     10010
Name: count, dtype: int64

Dataset loaded.


Cell 3 — Load tokenizer and E1 encoder weights The tokenizer must be the same tokenizer used by E1.

The encoder weights are initialized from E1's trained transformer. The E5 attention and classification heads are newly initialized because the document chunk-selection strategy and model training are changing.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    E1_DIR,
    local_files_only=True
)

encoder = AutoModel.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")
print("Encoder loaded from:", MODEL_NAME)

# Load the E1 checkpoint.
# E1 state may contain both encoder and classification-head parameters.
# We extract only the encoder weights.

e1_state = torch.load(
    E1_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

print("E1 checkpoint type:", type(e1_state))

if isinstance(e1_state, dict) and "state_dict" in e1_state:
    e1_state = e1_state["state_dict"]

if isinstance(e1_state, dict) and "model_state_dict" in e1_state:
    e1_state = e1_state["model_state_dict"]

# Find encoder parameter names.
encoder_state = {}

for key, value in e1_state.items():
    clean_key = key

    for prefix in ["module.", "model."]:
        if clean_key.startswith(prefix):
            clean_key = clean_key[len(prefix):]

    # Common E1 naming convention
    if clean_key.startswith("encoder."):
        encoder_state[clean_key[len("encoder."):]] = value

    elif clean_key.startswith("backbone."):
        encoder_state[clean_key[len("backbone."):]] = value

    elif clean_key.startswith("roberta."):
        encoder_state[clean_key[len("roberta."):]] = value

if len(encoder_state) == 0:
    print("WARNING: Could not identify encoder weights by prefix.")
    print("Checkpoint keys sample:")
    print(list(e1_state.keys())[:30])

    raise RuntimeError(
        "E1 encoder weights could not be extracted. "
        "Inspect E1 checkpoint keys and adapt the extraction."
    )

missing, unexpected = encoder.load_state_dict(
    encoder_state,
    strict=False
)

print("Encoder weights transferred from E1.")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

# Missing keys should be investigated.
# Ideally there should be no missing encoder parameters.

if missing:
    print("Missing parameter examples:", missing[:10])

encoder.to(DEVICE)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer loaded.
Encoder loaded from: xlm-roberta-base
E1 checkpoint type: <class 'collections.OrderedDict'>
Encoder weights transferred from E1.
Missing keys: 0
Unexpected keys: 0


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

Cell 4 — Document-wide chunk selection Unlike E1's original first-eight-chunks strategy, this implementation selects chunks evenly across the entire document.

For a document with more than eight chunks:

Keep the first chunk.
Keep the final chunk.
Select the remaining chunks at evenly spaced positions.
Preserve document order.

In [4]:
def make_document_chunks(
    text,
    tokenizer,
    max_len=512,
    stride=256,
    max_chunks=8
):
    """
    Tokenize a complete document and select chunks across
    the entire document.

    Returns:
        input_ids: [max_chunks, max_len]
        attention_mask: [max_chunks, max_len]
        valid_chunks: [max_chunks]
        selected_indices: list
    """

    text = str(text)

    token_ids = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    if len(token_ids) == 0:
        token_ids = []

    # Create all overlapping chunks.
    all_chunks = []

    if len(token_ids) == 0:
        all_chunks.append([])

    else:
        for start in range(0, len(token_ids), stride):
            chunk = token_ids[start:start + CONTENT_LEN]

            if len(chunk) == 0:
                break

            all_chunks.append(chunk)

            if start + CONTENT_LEN >= len(token_ids):
                break

    total_chunks = len(all_chunks)

    # Select chunks distributed across the document.
    if total_chunks <= max_chunks:
        selected_indices = list(range(total_chunks))

    else:
        selected_indices = np.linspace(
            0,
            total_chunks - 1,
            num=max_chunks
        ).round().astype(int).tolist()

        selected_indices = sorted(set(selected_indices))

        # Fill missing positions if rounding caused duplicates.
        if len(selected_indices) < max_chunks:
            remaining = [
                i for i in range(total_chunks)
                if i not in selected_indices
            ]

            for idx in remaining:
                selected_indices.append(idx)

                if len(selected_indices) == max_chunks:
                    break

            selected_indices = sorted(selected_indices)

    selected_chunks = [
        all_chunks[i] for i in selected_indices
    ]

    ids = []
    masks = []

    for chunk in selected_chunks:

        sequence = (
            [tokenizer.cls_token_id]
            + chunk
            + [tokenizer.sep_token_id]
        )

        mask = [1] * len(sequence)

        padding = max_len - len(sequence)

        sequence += [tokenizer.pad_token_id] * padding
        mask += [0] * padding

        ids.append(sequence)
        masks.append(mask)

    valid_count = len(ids)

    # Pad missing chunk slots.
    while len(ids) < max_chunks:
        ids.append([tokenizer.pad_token_id] * max_len)
        masks.append([0] * max_len)

    valid_chunks = [1] * valid_count + [0] * (
        max_chunks - valid_count
    )

    return (
        np.asarray(ids, dtype=np.int64),
        np.asarray(masks, dtype=np.int64),
        np.asarray(valid_chunks, dtype=np.int64),
        selected_indices
    )


# Sanity check
sample = X_train[0]

sample_ids, sample_masks, sample_valid, sample_indices = make_document_chunks(
    sample,
    tokenizer
)

print("Input IDs:", sample_ids.shape)
print("Attention masks:", sample_masks.shape)
print("Valid chunks:", sample_valid)
print("Selected chunk indices:", sample_indices)

assert sample_ids.shape == (MAX_CHUNKS, MAX_LEN)
assert sample_masks.shape == (MAX_CHUNKS, MAX_LEN)
assert sample_ids.dtype == np.int64

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2675 > 512). Running this sequence through the model will result in indexing errors


Input IDs: (8, 512)
Attention masks: (8, 512)
Valid chunks: [1 1 1 1 1 1 1 1]
Selected chunk indices: [0, 1, 3, 4, 5, 6, 8, 9]


Cell 5 — Prepare datasets To avoid repeatedly tokenizing the same documents during training, preprocess the chunk arrays once and retain them in memory.

In [5]:
class ChunkedDataset(Dataset):

    def __init__(self, texts, labels, tokenizer):
        self.labels = labels

        self.input_ids = []
        self.attention_masks = []
        self.valid_chunks = []

        for i, text in enumerate(texts):

            ids, masks, valid, _ = make_document_chunks(
                text,
                tokenizer
            )

            self.input_ids.append(ids)
            self.attention_masks.append(masks)
            self.valid_chunks.append(valid)

            if (i + 1) % 5000 == 0:
                print(
                    f"Tokenized {i + 1}/{len(texts)} documents"
                )

        self.input_ids = np.stack(self.input_ids)
        self.attention_masks = np.stack(self.attention_masks)
        self.valid_chunks = np.stack(self.valid_chunks)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(
                self.input_ids[idx],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                self.attention_masks[idx],
                dtype=torch.long
            ),
            "valid_chunks": torch.tensor(
                self.valid_chunks[idx],
                dtype=torch.bool
            ),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }


print("Preparing training dataset...")
train_dataset = ChunkedDataset(
    X_train,
    y_train,
    tokenizer
)

print("\nPreparing validation dataset...")
val_dataset = ChunkedDataset(
    X_val,
    y_val,
    tokenizer
)

print("\nPrepared:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Preparing training dataset...
Tokenized 5000/40000 documents
Tokenized 10000/40000 documents
Tokenized 15000/40000 documents
Tokenized 20000/40000 documents
Tokenized 25000/40000 documents
Tokenized 30000/40000 documents
Tokenized 35000/40000 documents
Tokenized 40000/40000 documents

Preparing validation dataset...
Tokenized 5000/40853 documents
Tokenized 10000/40853 documents
Tokenized 15000/40853 documents
Tokenized 20000/40853 documents
Tokenized 25000/40853 documents
Tokenized 30000/40853 documents
Tokenized 35000/40853 documents
Tokenized 40000/40853 documents

Prepared:
Train: 40000
Validation: 40853


Cell 6 — DataLoaders

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 20000
Validation batches: 20427


Cell 7 — E5 model architecture The model:

Encodes each chunk using XLM-R.
Extracts the CLS representation of every chunk.
Uses learned attention pooling across valid chunks.
Applies dropout and a three-class classification head.

In [7]:
class E5ChunkedClassifier(nn.Module):

    def __init__(
        self,
        encoder,
        hidden_size=768,
        dropout=0.2
    ):
        super().__init__()

        self.encoder = encoder

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1)
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_size,
            3
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        valid_chunks
    ):

        B, C, L = input_ids.shape

        input_ids = input_ids.view(B * C, L)
        attention_mask = attention_mask.view(B * C, L)

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embeddings = outputs.last_hidden_state[:, 0, :]

        cls_embeddings = cls_embeddings.view(
            B, C, -1
        )

        # Calculate attention scores.
        scores = self.attention(
            cls_embeddings
        ).squeeze(-1)

        # Exclude padded chunks.
        scores = scores.masked_fill(
            ~valid_chunks,
            -1e4
        )

        weights = torch.softmax(scores, dim=1)

        # Weighted sum of chunk embeddings.
        document_embedding = torch.sum(
            cls_embeddings * weights.unsqueeze(-1),
            dim=1
        )

        document_embedding = self.dropout(
            document_embedding
        )

        logits = self.classifier(
            document_embedding
        )

        return logits


model = E5ChunkedClassifier(
    encoder=encoder,
    hidden_size=encoder.config.hidden_size,
    dropout=0.2
).to(DEVICE)

print("E5 model initialized.")

E5 model initialized.


Cell 8 — Class weights, optimizer and scheduler The class weights are calculated exclusively from training labels.

Separate learning rates are used for the pretrained encoder and the newly initialized attention/classification heads.

In [8]:
class_counts = np.bincount(
    y_train,
    minlength=3
)

class_weights = len(y_train) / (
    3.0 * np.maximum(class_counts, 1)
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.encoder.parameters(),
            "lr": ENCODER_LR
        },
        {
            "params": list(model.attention.parameters())
                      + list(model.classifier.parameters()),
            "lr": HEAD_LR
        }
    ],
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8
)

updates_per_epoch = int(
    np.ceil(len(train_loader) / GRAD_ACCUM)
)

total_steps = updates_per_epoch * EPOCHS

warmup_steps = int(
    WARMUP_RATIO * total_steps
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Total optimizer steps:", total_steps)
print("Warmup steps:", warmup_steps)

Class counts: [ 9801 17909 12290]
Class weights: tensor([1.3604, 0.7445, 1.0849], device='cuda:0')
Total optimizer steps: 15000
Warmup steps: 900


Cell 9 — Training and validation functions

In [9]:
from contextlib import nullcontext

def autocast_context():
    if USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        )
    return nullcontext()


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


def train_one_epoch(model, loader):

    model.train()

    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):

        input_ids = batch["input_ids"].to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True
        )

        valid_chunks = batch["valid_chunks"].to(
            DEVICE,
            dtype=torch.bool,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True
        )

        assert input_ids.shape[-1] == MAX_LEN
        assert input_ids.dtype == torch.long

        with autocast_context():

            logits = model(
                input_ids,
                attention_mask,
                valid_chunks
            )

            loss = criterion(logits, labels)
            loss = loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        total_loss += loss.item() * GRAD_ACCUM

        if (
            (step + 1) % GRAD_ACCUM == 0
            or (step + 1) == len(loader)
        ):

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

            scheduler.step()

        if (step + 1) % 100 == 0:
            print(
                f"Training step {step + 1}/{len(loader)} "
                f"loss={total_loss / (step + 1):.4f}"
            )

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    all_logits = []
    all_labels = []

    total_loss = 0.0

    for batch in loader:

        input_ids = batch["input_ids"].to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True
        )

        valid_chunks = batch["valid_chunks"].to(
            DEVICE,
            dtype=torch.bool,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True
        )

        with autocast_context():

            logits = model(
                input_ids,
                attention_mask,
                valid_chunks
            )

            loss = criterion(logits, labels)

        total_loss += loss.item()

        all_logits.append(
            logits.float().cpu().numpy()
        )

        all_labels.append(
            labels.cpu().numpy()
        )

    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)

    probs = torch.softmax(
        torch.tensor(all_logits),
        dim=1
    ).numpy()

    preds = probs.argmax(axis=1)

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro"
    )

    return {
        "loss": total_loss / len(loader),
        "macro_f1": macro_f1,
        "logits": all_logits,
        "probs": probs,
        "preds": preds,
        "labels": all_labels
    }

Cell 10 — Train E5 and save the best checkpoint The best checkpoint is selected using validation macro-F1. The model is saved after every improvement so that the best result survives a runtime interruption.

In [10]:
best_macro_f1 = -1.0
best_epoch = -1
epochs_without_improvement = 0

history = []

for epoch in range(EPOCHS):

    print("\n" + "=" * 70)
    print(f"E5 EPOCH {epoch + 1}/{EPOCHS}")
    print("=" * 70)

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    val_result = evaluate(
        model,
        val_loader
    )

    print(f"\nTrain loss: {train_loss:.5f}")
    print(f"Validation loss: {val_result['loss']:.5f}")
    print(f"Validation macro-F1: {val_result['macro_f1']:.6f}")

    history.append({
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_result["loss"]),
        "val_macro_f1": float(val_result["macro_f1"])
    })

    # Save best checkpoint.
    if val_result["macro_f1"] > best_macro_f1:

        best_macro_f1 = val_result["macro_f1"]
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            f"{OUT_DIR}/e5_best_model.pt"
        )

        np.save(
            f"{OUT_DIR}/e5_best_val_probs.npy",
            val_result["probs"]
        )

        np.save(
            f"{OUT_DIR}/e5_best_val_logits.npy",
            val_result["logits"]
        )

        np.save(
            f"{OUT_DIR}/e5_best_val_preds.npy",
            val_result["preds"]
        )

        print("<-- Best checkpoint saved.")

    else:
        epochs_without_improvement += 1
        print("No validation improvement.")

    # Save history after each epoch.
    pd.DataFrame(history).to_csv(
        f"{OUT_DIR}/training_history.csv",
        index=False
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

print("\nTraining finished.")
print("Best epoch:", best_epoch)
print("Best validation macro-F1:", best_macro_f1)


E5 EPOCH 1/3
Training step 100/20000 loss=1.2673
Training step 200/20000 loss=1.2155
Training step 300/20000 loss=1.1639
Training step 400/20000 loss=1.0774
Training step 500/20000 loss=0.9988
Training step 600/20000 loss=0.9269
Training step 700/20000 loss=0.8560
Training step 800/20000 loss=0.7944
Training step 900/20000 loss=0.7359
Training step 1000/20000 loss=0.6887
Training step 1100/20000 loss=0.6401
Training step 1200/20000 loss=0.6025
Training step 1300/20000 loss=0.5673
Training step 1400/20000 loss=0.5438
Training step 1500/20000 loss=0.5186
Training step 1600/20000 loss=0.4981
Training step 1700/20000 loss=0.4883
Training step 1800/20000 loss=0.4720
Training step 1900/20000 loss=0.4553
Training step 2000/20000 loss=0.4437
Training step 2100/20000 loss=0.4323
Training step 2200/20000 loss=0.4259
Training step 2300/20000 loss=0.4176
Training step 2400/20000 loss=0.4058
Training step 2500/20000 loss=0.3983
Training step 2600/20000 loss=0.3924
Training step 2700/20000 loss=0.3

Cell 11 — Load best model and generate final validation metrics

In [11]:
best_state = torch.load(
    f"{OUT_DIR}/e5_best_model.pt",
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(best_state)
model.eval()

final_result = evaluate(
    model,
    val_loader
)

val_probs = final_result["probs"]
val_preds = final_result["preds"]

print("Final validation macro-F1:", final_result["macro_f1"])

# Save final probabilities and predictions.
np.save(
    f"{OUT_DIR}/validation_probabilities.npy",
    val_probs
)

np.save(
    f"{OUT_DIR}/validation_predictions.npy",
    val_preds
)

np.save(
    f"{OUT_DIR}/validation_labels.npy",
    y_val
)

if "id" in val_df.columns:
    np.save(
        f"{OUT_DIR}/validation_ids.npy",
        val_df["id"].to_numpy()
    )

Final validation macro-F1: 0.9172033438926294


Cell 12 — Precision, recall, F1 table

In [12]:
report = classification_report(
    y_val,
    val_preds,
    labels=[0, 1, 2],
    target_names=LABELS,
    output_dict=True,
    zero_division=0
)

precision_table = pd.DataFrame(report).T

print("E5 validation precision table:")
display(precision_table.round(5))

precision_table.to_csv(
    f"{OUT_DIR}/e5_precision_table.csv"
)

# Confusion matrix
cm = confusion_matrix(
    y_val,
    val_preds,
    labels=[0, 1, 2]
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{x}" for x in LABELS],
    columns=[f"Pred_{x}" for x in LABELS]
)

print("E5 confusion matrix:")
display(cm_df)

cm_df.to_csv(
    f"{OUT_DIR}/e5_confusion_matrix.csv"
)

E5 validation precision table:


,precision,recall,f1-score,support
human,0.94305,0.79081,0.86025,10010.00000
ai,0.98668,0.99639,0.99151,18291.00000
fusion,0.85366,0.95132,0.89985,12552.00000
accuracy,0.93217,0.93217,0.93217,0.93217
macro avg,0.92780,0.91284,0.91720,40853.00000
weighted avg,0.93512,0.93217,0.93119,40853.00000


E5 confusion matrix:


,Pred_human,Pred_ai,Pred_fusion
True_human,7916,92,2002
True_ai,21,18225,45
True_fusion,457,154,11941


Cell 13 — Save tokenizer and model configuration

In [13]:
tokenizer.save_pretrained(
    f"{OUT_DIR}/tokenizer"
)

config = {
    "experiment": "E5_Better_Document_Wide_Chunking",

    "base_model": MODEL_NAME,
    "initialized_from_e1": True,

    "max_len": MAX_LEN,
    "content_len": CONTENT_LEN,
    "stride": STRIDE,
    "max_chunks": MAX_CHUNKS,

    "chunk_selection": "uniform_document_wide",

    "pooling": "learned_attention",

    "dropout": 0.2,
    "num_classes": 3,
    "class_order": LABELS,

    "epochs_requested": EPOCHS,
    "best_epoch": best_epoch,

    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,

    "encoder_lr": ENCODER_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,

    "warmup_ratio": WARMUP_RATIO,
    "max_grad_norm": MAX_GRAD_NORM,

    "validation_macro_f1": float(final_result["macro_f1"]),

    "seed": SEED
}

with open(
    f"{OUT_DIR}/e5_config.json",
    "w"
) as f:
    json.dump(config, f, indent=4)

print("Tokenizer and configuration saved.")

Tokenizer and configuration saved.


Cell 14 — Save final experiment summary

In [14]:
summary = {
    "experiment": "E5",
    "description": "Improved document-wide chunk selection",
    "best_epoch": int(best_epoch),
    "validation_macro_f1": float(final_result["macro_f1"]),
    "validation_loss": float(final_result["loss"]),
    "class_order": LABELS,
    "training_rows": len(train_df),
    "validation_rows": len(val_df)
}

with open(
    f"{OUT_DIR}/experiment_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

print("\nE5 finished successfully.")
print("Artifacts saved to:", OUT_DIR)

print("\nSaved files:")
for filename in sorted(os.listdir(OUT_DIR)):
    filepath = os.path.join(OUT_DIR, filename)

    if os.path.isfile(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"{filename:40s} {size_mb:8.2f} MB")


E5 finished successfully.
Artifacts saved to: /kaggle/working/e5_chunking_artifacts

Saved files:
e5_best_model.pt                          1061.87 MB
e5_best_val_logits.npy                       0.47 MB
e5_best_val_preds.npy                        0.31 MB
e5_best_val_probs.npy                        0.47 MB
e5_config.json                               0.00 MB
e5_confusion_matrix.csv                      0.00 MB
e5_precision_table.csv                       0.00 MB
experiment_summary.json                      0.00 MB
training_history.csv                         0.00 MB
validation_ids.npy                           0.35 MB
validation_labels.npy                        0.31 MB
validation_predictions.npy                   0.31 MB
validation_probabilities.npy                 0.47 MB
